In [0]:
from pyspark.sql import functions as F
# Lectura de tabla inicial
df_silver = spark.table("citibike.01_bronze.citibikes_raw")
display(df_silver.limit(10))

# Duplicados
print('\n Before duplicates count')
display(df_silver.count())

print('\n After deduplicating count')
df_deduplicated = df_silver.dropDuplicates()
display(df_deduplicated.count())

# Nulos
print('\n Nulls by column')
df_deduplicated.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_deduplicated.columns
]).show()

df_nulls_removed = df_deduplicated.dropna(subset=["end_station_id"])

print('\n After nulls removed')
df_nulls_removed.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_nulls_removed.columns
]).show()

# Cambio de tipados
df_typed = (df_nulls_removed
            .withColumn("started_at", F.to_timestamp("started_at", "yyyy-MM-dd HH:mm:ss.SSS"))
            .withColumn("ended_at", F.to_timestamp("ended_at", "yyyy-MM-dd HH:mm:ss.SSS"))
            .withColumn("start_lat", F.col("start_lat").cast("double"))
            .withColumn("start_lng", F.col("start_lng").cast("double"))
            .withColumn("end_lat", F.col("end_lat").cast("double"))
            .withColumn("end_lng", F.col("end_lng").cast("double"))
)

display(df_typed.limit(10))

df_typed.write.mode("overwrite").saveAsTable("citibike.02_silver.citibikes")